In [0]:
from typing import Any

In [0]:
import logging

from py4j.java_gateway import JavaObject  # type: ignore
from pyspark.sql import SparkSession
from pyspark.sql.functions import (
    concat_ws,
    current_timestamp,
    from_utc_timestamp,
    lit,
    sha2,
)

In [0]:
dbutils.widgets.text('curated_zone', 'None', '01-Curated_zone')
dbutils.widgets.text('catalog_landed', 'None', '02-Catalog_landed')
dbutils.widgets.text('reference_zone', 'None', '03-Reference_zone')
dbutils.widgets.text('run_date', 'None', '04-Run Date')
dbutils.widgets.text('schema', 'None', '05-Schema')
dbutils.widgets.text('env_prefix', 'None', '06-Environment')
curated_zone = dbutils.widgets.get('curated_zone')
catalog_landed = dbutils.widgets.get('catalog_landed')
reference_zone = dbutils.widgets.get('reference_zone')
run_date = dbutils.widgets.get('run_date')
schema = dbutils.widgets.get('schema')
env_prefix = dbutils.widgets.get('env_prefix')

import sys, os
relative_path = '../../helpers/lng_lnd'
path = os.path.abspath(relative_path)
if path not in sys.path:
  sys.path.append(path)
from landing_helpers import *
from landed_helpers import *
relative_path = '../../helpers/curated_helpers'
path = os.path.abspath(relative_path)
if path not in sys.path:
  sys.path.append(path)
from delta_table_creator import *
from extract import *
from transform import *
spark = SparkSession.builder.getOrCreate()


In [0]:
def load_curated_firm_structure_whl(
    spark, curated_zone, catalog_landed, reference_zone, run_date, schema, env_prefix
) -> None:
    args = [curated_zone, catalog_landed, reference_zone, run_date, schema, env_prefix]
    if all(item is not None for item in args) :
        date_filter = f"businessdate = {run_date!r}"

        # schema = (
        #     "FirmStructureId string,"
        #     "SourceSystem string,"
        #     "IngestionTimestamp timestamp,"
        #     "BatchId string,"
        #     "FirmStructureKey string,"
        #     "EmployeePartyKey string,"
        #     "EmployeeCategory string,"
        #     "EmployeeFirstName string,"
        #     "EmployeeLastName string,"
        #     "EmployeeFullName string,"
        #     "EmployeeEmail string,"
        #     "EmployeeUsername string,"
        #     "EmployeeUserTypeCode string,"
        #     "EmployeeUserType string,"
        #     "EmployeePositionCode string ,"
        #     "EmployeePosition string,"
        #     "DepartmentCode string,"
        #     "Department string,"
        #     "TeamCode string,"
        #     "Team string,"
        #     "SupervisorPartyKey string,"
        #     "SupervisorPartyName string,"
        #     "SupervisorUsername string,"
        #     "ManagerPartyKey string,"
        #     "ManagerPartyName string,"
        #     "ManagerUsername string,"
        #     "DirectorPartyKey string,"
        #     "DirectorPartyName string,"
        #     "DirectorUsername string,"
        #     "IsEmployeeActiveFlag boolean,"
        #     "CreatedDate date,"
        #     "LastUpdatedDate date,"
        #     "BusinessDate date"
        # )

        cluster_columns = ["BusinessDate"]

        curated_table_path = f"{curated_zone}.FirmStructure"

        # ================== Create Delta Table ==================
        logging.info("Creating Delta Table...")
        if spark.catalog.tableExists(curated_table_path) is False:
            create_delta_table(
                spark,
                curated_table_path,
                cluster_columns,
                schema,  # type: ignore
            )
        else:
            logging.info("Table already exists.")

        # ================== Delete From Delta Table ==================
        if spark.catalog.tableExists(curated_table_path) is True:
            delete_from_curated_table(
                spark,
                curated_table_path,
                run_date,  # type: ignore
                "whl",
            )        
        else:
            logging.info("Tables does not exist, no need to delete entries for business date!")

        # ================== MainAssignment ==================

        MainAssignment_extra_filters = None

        if MainAssignment_extra_filters is not None:
            MainAssignment_extra_filters.append(date_filter)
            MainAssignment_final_filters = " AND ".join(MainAssignment_extra_filters)
        else:
            MainAssignment_final_filters = date_filter

        MainAssignment_selected_fields = [
            "Officer",
            "Supervisor",
            "AssistantManager",
            "Manager",
            "Head",
            "cdi",
            "businessdate",
        ]

        MainAssignment = load_landed_delta_table(
            spark,
            catalog_landed,  # type: ignore
            "whl_MainAssignment",
            MainAssignment_final_filters,
            MainAssignment_selected_fields,
        )

        # ================== firm_structure ==================
        firm_structure_officer_primary_key_fields = [
            MainAssignment.Officer,
            lit("Employee"),
            lit("whl"),
        ]
        FirmStructureKey_officer_primary_key_fields = [
            MainAssignment.Officer,
            lit("Employee"),
        ]

        firm_structure_officer = (
            MainAssignment.withColumn(
                "FirmStructureId",
                sha2(
                    concat_ws(";", *firm_structure_officer_primary_key_fields),
                    256,
                ),
            )
        ).withColumn(
            "FirmStructureKey",
            concat_ws(";", *FirmStructureKey_officer_primary_key_fields),
        )

        firm_structure_officer = firm_structure_officer.select(
            firm_structure_officer.FirmStructureId.alias("FirmStructureId"),
            lit("whl").alias("SourceSystem"),
            lit(from_utc_timestamp(current_timestamp(), "Europe/Athens")).alias(
                "IngestionTimestamp"
            ),
            lit(None).alias("BatchId"),
            firm_structure_officer.FirmStructureKey.cast("string").alias(
                "FirmStructureKey"
            ),
            firm_structure_officer.FirmStructureKey.cast("string").alias(
                "EmployeePartyKey"
            ),
            lit("Employee").cast("string").alias("EmployeeCategory"),
            lit(None).cast("string").alias("EmployeeFirstName"),
            lit(None).cast("string").alias("EmployeeLastName"),
            lit(None).cast("string").alias("EmployeeFullName"),
            lit(None).cast("string").alias("EmployeeEmail"),
            firm_structure_officer.Officer.cast("string").alias("EmployeeUsername"),
            lit(None).cast("string").alias("EmployeeUserTypeCode"),
            lit(None).cast("string").alias("EmployeeUserType"),
            lit(None).cast("string").alias("EmployeePositionCode"),
            lit(None).cast("string").alias("EmployeePosition"),
            lit(None).cast("string").alias("DepartmentCode"),
            lit(None).cast("string").alias("Department"),
            lit(None).cast("string").alias("TeamCode"),
            lit(None).cast("string").alias("Team"),
            lit(None).cast("string").alias("SupervisorPartyKey"),
            lit(None).cast("string").alias("SupervisorPartyName"),
            lit(None).cast("string").alias("SupervisorUsername"),
            lit(None).cast("string").alias("ManagerPartyKey"),
            lit(None).cast("string").alias("ManagerPartyName"),
            lit(None).cast("string").alias("ManagerUsername"),
            lit(None).cast("string").alias("DirectorPartyKey"),
            lit(None).cast("string").alias("DirectorPartyName"),
            lit(None).cast("string").alias("DirectorUsername"),
            lit(None).cast("boolean").alias("IsEmployeeActiveFlag"),
            lit(None).cast("date").alias("CreatedDate"),
            lit(None).cast("date").alias("LastUpdatedDate"),
            firm_structure_officer.businessdate.cast("date").alias("BusinessDate"),
        ).distinct()
        firm_structure_supervisor_primary_key_fields = [
            MainAssignment.Supervisor,
            lit("Employee"),
            lit("whl"),
        ]
        FirmStructureKey_supervisor_primary_key_fields = [
            MainAssignment.Supervisor,
            lit("Employee"),
        ]

        firm_structure_supervisor = (
            MainAssignment.withColumn(
                "FirmStructureId",
                sha2(
                    concat_ws(";", *firm_structure_supervisor_primary_key_fields),
                    256,
                ),
            )
        ).withColumn(
            "FirmStructureKey",
            concat_ws(";", *FirmStructureKey_supervisor_primary_key_fields),
        )
        firm_structure_supervisor = firm_structure_supervisor.select(
            firm_structure_supervisor.FirmStructureId.alias("FirmStructureId"),
            lit("whl").alias("SourceSystem"),
            lit(from_utc_timestamp(current_timestamp(), "Europe/Athens")).alias(
                "IngestionTimestamp"
            ),
            lit(None).alias("BatchId"),
            firm_structure_supervisor.FirmStructureKey.cast("string").alias(
                "FirmStructureKey"
            ),
            firm_structure_supervisor.FirmStructureKey.cast("string").alias(
                "EmployeePartyKey"
            ),
            lit("Employee").cast("string").alias("EmployeeCategory"),
            lit(None).cast("string").alias("EmployeeFirstName"),
            lit(None).cast("string").alias("EmployeeLastName"),
            lit(None).cast("string").alias("EmployeeFullName"),
            lit(None).cast("string").alias("EmployeeEmail"),
            firm_structure_supervisor.Supervisor.cast("string").alias(
                "EmployeeUsername"
            ),
            lit(None).cast("string").alias("EmployeeUserTypeCode"),
            lit(None).cast("string").alias("EmployeeUserType"),
            lit(None).cast("string").alias("EmployeePositionCode"),
            lit(None).cast("string").alias("EmployeePosition"),
            lit(None).cast("string").alias("DepartmentCode"),
            lit(None).cast("string").alias("Department"),
            lit(None).cast("string").alias("TeamCode"),
            lit(None).cast("string").alias("Team"),
            lit(None).cast("string").alias("SupervisorPartyKey"),
            lit(None).cast("string").alias("SupervisorPartyName"),
            lit(None).cast("string").alias("SupervisorUsername"),
            lit(None).cast("string").alias("ManagerPartyKey"),
            lit(None).cast("string").alias("ManagerPartyName"),
            lit(None).cast("string").alias("ManagerUsername"),
            lit(None).cast("string").alias("DirectorPartyKey"),
            lit(None).cast("string").alias("DirectorPartyName"),
            lit(None).cast("string").alias("DirectorUsername"),
            lit(None).cast("boolean").alias("IsEmployeeActiveFlag"),
            lit(None).cast("date").alias("CreatedDate"),
            lit(None).cast("date").alias("LastUpdatedDate"),
            firm_structure_supervisor.businessdate.cast("date").alias("BusinessDate"),
        ).distinct()
        firm_structure_AssistantManager_primary_key_fields = [
            MainAssignment.AssistantManager,
            lit("Employee"),
            lit("whl"),
        ]
        FirmStructureKey_AssistantManager_primary_key_fields = [
            MainAssignment.AssistantManager,
            lit("Employee"),
        ]

        firm_structure_assistant_manager = (
            MainAssignment.withColumn(
                "FirmStructureId",
                sha2(
                    concat_ws(";", *firm_structure_AssistantManager_primary_key_fields),
                    256,
                ),
            )
        ).withColumn(
            "FirmStructureKey",
            concat_ws(";", *FirmStructureKey_AssistantManager_primary_key_fields),
        )
        firm_structure_assistant_manager = firm_structure_assistant_manager.select(
            firm_structure_assistant_manager.FirmStructureId.alias("FirmStructureId"),
            lit("whl").alias("SourceSystem"),
            lit(from_utc_timestamp(current_timestamp(), "Europe/Athens")).alias(
                "IngestionTimestamp"
            ),
            lit(None).alias("BatchId"),
            firm_structure_assistant_manager.FirmStructureKey.cast("string").alias(
                "FirmStructureKey"
            ),
            firm_structure_assistant_manager.FirmStructureKey.cast("string").alias(
                "EmployeePartyKey"
            ),
            lit("Employee").cast("string").alias("EmployeeCategory"),
            lit(None).cast("string").alias("EmployeeFirstName"),
            lit(None).cast("string").alias("EmployeeLastName"),
            lit(None).cast("string").alias("EmployeeFullName"),
            lit(None).cast("string").alias("EmployeeEmail"),
            firm_structure_assistant_manager.AssistantManager.cast("string").alias(
                "EmployeeUsername"
            ),
            lit(None).cast("string").alias("EmployeeUserTypeCode"),
            lit(None).cast("string").alias("EmployeeUserType"),
            lit(None).cast("string").alias("EmployeePositionCode"),
            lit(None).cast("string").alias("EmployeePosition"),
            lit(None).cast("string").alias("DepartmentCode"),
            lit(None).cast("string").alias("Department"),
            lit(None).cast("string").alias("TeamCode"),
            lit(None).cast("string").alias("Team"),
            lit(None).cast("string").alias("SupervisorPartyKey"),
            lit(None).cast("string").alias("SupervisorPartyName"),
            lit(None).cast("string").alias("SupervisorUsername"),
            lit(None).cast("string").alias("ManagerPartyKey"),
            lit(None).cast("string").alias("ManagerPartyName"),
            lit(None).cast("string").alias("ManagerUsername"),
            lit(None).cast("string").alias("DirectorPartyKey"),
            lit(None).cast("string").alias("DirectorPartyName"),
            lit(None).cast("string").alias("DirectorUsername"),
            lit(None).cast("boolean").alias("IsEmployeeActiveFlag"),
            lit(None).cast("date").alias("CreatedDate"),
            lit(None).cast("date").alias("LastUpdatedDate"),
            firm_structure_assistant_manager.businessdate.cast("date").alias(
                "BusinessDate"
            ),
        ).distinct()
        firm_structure_Manager_primary_key_fields = [
            MainAssignment.Manager,
            lit("Employee"),
            lit("whl"),
        ]
        FirmStructureKey_Manager_primary_key_fields = [
            MainAssignment.Manager,
            lit("Employee"),
        ]

        firm_structure_manager = (
            MainAssignment.withColumn(
                "FirmStructureId",
                sha2(
                    concat_ws(";", *firm_structure_Manager_primary_key_fields),
                    256,
                ),
            )
        ).withColumn(
            "FirmStructureKey",
            concat_ws(";", *FirmStructureKey_Manager_primary_key_fields),
        )
        firm_structure_manager = firm_structure_manager.select(
            firm_structure_manager.FirmStructureId.alias("FirmStructureId"),
            lit("whl").alias("SourceSystem"),
            lit(from_utc_timestamp(current_timestamp(), "Europe/Athens")).alias(
                "IngestionTimestamp"
            ),
            lit(None).alias("BatchId"),
            firm_structure_manager.FirmStructureKey.cast("string").alias(
                "FirmStructureKey"
            ),
            firm_structure_manager.FirmStructureKey.cast("string").alias(
                "EmployeePartyKey"
            ),
            lit("Employee").cast("string").alias("EmployeeCategory"),
            lit(None).cast("string").alias("EmployeeFirstName"),
            lit(None).cast("string").alias("EmployeeLastName"),
            lit(None).cast("string").alias("EmployeeFullName"),
            lit(None).cast("string").alias("EmployeeEmail"),
            firm_structure_manager.Manager.cast("string").alias("EmployeeUsername"),
            lit(None).cast("string").alias("EmployeeUserTypeCode"),
            lit(None).cast("string").alias("EmployeeUserType"),
            lit(None).cast("string").alias("EmployeePositionCode"),
            lit(None).cast("string").alias("EmployeePosition"),
            lit(None).cast("string").alias("DepartmentCode"),
            lit(None).cast("string").alias("Department"),
            lit(None).cast("string").alias("TeamCode"),
            lit(None).cast("string").alias("Team"),
            lit(None).cast("string").alias("SupervisorPartyKey"),
            lit(None).cast("string").alias("SupervisorPartyName"),
            lit(None).cast("string").alias("SupervisorUsername"),
            lit(None).cast("string").alias("ManagerPartyKey"),
            lit(None).cast("string").alias("ManagerPartyName"),
            lit(None).cast("string").alias("ManagerUsername"),
            lit(None).cast("string").alias("DirectorPartyKey"),
            lit(None).cast("string").alias("DirectorPartyName"),
            lit(None).cast("string").alias("DirectorUsername"),
            lit(None).cast("boolean").alias("IsEmployeeActiveFlag"),
            lit(None).cast("date").alias("CreatedDate"),
            lit(None).cast("date").alias("LastUpdatedDate"),
            firm_structure_manager.businessdate.cast("date").alias("BusinessDate"),
        ).distinct()
        firm_structure_Head_primary_key_fields = [
            MainAssignment.Head,
            lit("Employee"),
            lit("whl"),
        ]
        FirmStructureKey_Head_primary_key_fields = [
            MainAssignment.Head,
            lit("Employee"),
        ]

        firm_structure_Head = (
            MainAssignment.withColumn(
                "FirmStructureId",
                sha2(
                    concat_ws(";", *firm_structure_Head_primary_key_fields),
                    256,
                ),
            )
        ).withColumn(
            "FirmStructureKey",
            concat_ws(";", *FirmStructureKey_Head_primary_key_fields),
        )
        firm_structure_Head = firm_structure_Head.select(
            firm_structure_Head.FirmStructureId.alias("FirmStructureId"),
            lit("whl").alias("SourceSystem"),
            lit(from_utc_timestamp(current_timestamp(), "Europe/Athens")).alias(
                "IngestionTimestamp"
            ),
            lit(None).alias("BatchId"),
            firm_structure_Head.FirmStructureKey.cast("string").alias(
                "FirmStructureKey"
            ),
            firm_structure_Head.FirmStructureKey.cast("string").alias(
                "EmployeePartyKey"
            ),
            lit("Employee").cast("string").alias("EmployeeCategory"),
            lit(None).cast("string").alias("EmployeeFirstName"),
            lit(None).cast("string").alias("EmployeeLastName"),
            lit(None).cast("string").alias("EmployeeFullName"),
            lit(None).cast("string").alias("EmployeeEmail"),
            firm_structure_Head.Head.cast("string").alias("EmployeeUsername"),
            lit(None).cast("string").alias("EmployeeUserTypeCode"),
            lit(None).cast("string").alias("EmployeeUserType"),
            lit(None).cast("string").alias("EmployeePositionCode"),
            lit(None).cast("string").alias("EmployeePosition"),
            lit(None).cast("string").alias("DepartmentCode"),
            lit(None).cast("string").alias("Department"),
            lit(None).cast("string").alias("TeamCode"),
            lit(None).cast("string").alias("Team"),
            lit(None).cast("string").alias("SupervisorPartyKey"),
            lit(None).cast("string").alias("SupervisorPartyName"),
            lit(None).cast("string").alias("SupervisorUsername"),
            lit(None).cast("string").alias("ManagerPartyKey"),
            lit(None).cast("string").alias("ManagerPartyName"),
            lit(None).cast("string").alias("ManagerUsername"),
            lit(None).cast("string").alias("DirectorPartyKey"),
            lit(None).cast("string").alias("DirectorPartyName"),
            lit(None).cast("string").alias("DirectorUsername"),
            lit(None).cast("boolean").alias("IsEmployeeActiveFlag"),
            lit(None).cast("date").alias("CreatedDate"),
            lit(None).cast("date").alias("LastUpdatedDate"),
            firm_structure_Head.businessdate.cast("date").alias("BusinessDate"),
        ).distinct()

        firm_structure = (
            firm_structure_officer.union(firm_structure_supervisor)
            .distinct()
            .union(firm_structure_assistant_manager)
            .distinct()
            .union(firm_structure_manager)
            .distinct()
            .union(firm_structure_Head)
            .distinct()
        )
        firm_structure.write.format("delta").clusterBy("BusinessDate").mode(
            "append"
        ).saveAsTable(curated_table_path)

        # ========================== Append ==========================

    else:
        null_args = [
            arg_name
            for arg_name, arg in args
            if arg
            in [
                curated_zone,
                catalog_landed,
                reference_zone,
                run_date,
                schema,
                env_prefix,
            ]
            and arg is None
        ]

        for arg_name in null_args:
            logging.error(f"No {arg_name} specified!")

    logging.info("We did it!")


In [0]:
load_curated_firm_structure_whl(spark, curated_zone, catalog_landed, reference_zone, run_date, schema, env_prefix)